# 04 — Leakage-controlled observed-traffic benchmark

This notebook reads the strict local artifact bundle produced from the verified 2024 TfC counts. It does not train on notebook state, refit after inspecting the holdout, download data, or generate replacement observations. Artifact hashes, source labels, split manifests, and the scientific gate are validated before metrics are displayed.

The study is a bounded research prototype. It is not a deployed traffic controller, a calibrated Greater Cairo digital twin, or a causal study of Earth observation and traffic.


## Execute the frozen workflow on Windows PowerShell 5.1

Prepare the local bundle as shown in notebook 03, then run from the repository root with Python 3.12:

```powershell
Set-Location "D:\UrbanTwin-AI\UrbanTwin-AI"
.\.venv\Scripts\Activate.ps1
urbantwin observed-study `
  --data-dir "data\observed\tfc\normalized_v1" `
  --output-dir "results\observed_traffic" `
  --config "configs\observed_traffic.yaml" `
  --eo-dir "data\real"
```

Optional local MLflow logging is additive and does not replace the authoritative artifact bundle:

```powershell
python -m pip install -e ".[mlflow]"
urbantwin observed-study `
  --data-dir "data\observed\tfc\normalized_v1" `
  --output-dir "results\observed_traffic" `
  --config "configs\observed_traffic.yaml" `
  --eo-dir "data\real" `
  --mlflow
```

The command refuses to overwrite an existing research bundle. Use a new output directory for a new protocol or configuration.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urbantwin.observed.artifacts import validate_observed_artifacts

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter from the UrbanTwin-AI repository root.")

RESULTS_DIR = ROOT / "results/observed_traffic"
if not (RESULTS_DIR / "provenance.json").is_file():
    raise FileNotFoundError(
        "The verified observed benchmark is missing. Run the PowerShell command above."
    )
bundle = validate_observed_artifacts(ROOT, RESULTS_DIR)
metrics = bundle["metrics"].copy()
cv_metrics = bundle["cv_metrics"].copy()
predictions = bundle["predictions"].copy()
site_errors = bundle["site_errors"].copy()
provenance = bundle["provenance"]
split = bundle["split_manifest"]

if provenance["data_kind"] != "real_observed_traffic_analysis":
    raise ValueError("Observed benchmark source label is incompatible.")
if provenance["study_gate"]["allow_measured_demand_sumo"] is not False:
    raise ValueError("Unexpected signal-simulation gate state.")


## Frozen evaluation protocol

Every exact directional segment appears on one date only. The last 20% of distinct survey dates therefore form a **joint future-date and unseen-inferred-site** holdout. Date and site effects are confounded, so this must not be described as an isolated temporal-generalization test. Model development uses five site-grouped folds inside the earlier dates; the holdout is evaluated once.


In [ ]:
split_summary = pd.Series(
    {
        "protocol": split["protocol"],
        "training dates": len(split["train_dates"]),
        "holdout dates": len(split["test_dates"]),
        "training rows": split["train_rows"],
        "holdout rows": split["test_rows"],
        "training inferred sites": split["train_inferred_sites"],
        "holdout inferred sites": split["test_inferred_sites"],
        "site overlap": len(split["site_overlap"]),
        "excluded ambiguous OSM-join rows": split["excluded_ambiguous_rows"],
    },
    name="value",
)
split_summary.to_frame()


Leakage controls exclude current-period class counts, current totals, paired-direction counts, IDs, names, exact dates, and future rows from model inputs. Five segment joins with conflicting OSM IDs are excluded from the primary benchmark. The 60 analytical site groups are reproducible 250 m geometry clusters, not publisher-supplied location IDs.


## Measurement-free volume estimation

These models estimate directional 20-minute total vehicle volume using only clock/calendar descriptors and static released road geometry attributes available without a current traffic measurement. Bootstrap intervals resample inferred holdout sites, not individual rows.


In [ ]:
measurement_free = (
    metrics.loc[metrics["information_budget"] == "measurement_free_estimation"]
    .sort_values("mae")
    .reset_index(drop=True)
)
measurement_free[
    [
        "model",
        "feature_set",
        "rows",
        "inferred_sites",
        "mae",
        "mae_ci95_low",
        "mae_ci95_high",
        "rmse",
        "rmse_ci95_low",
        "rmse_ci95_high",
        "mean_bias",
        "mean_poisson_deviance",
    ]
]


In [ ]:
labels = (measurement_free["model"] + " / " + measurement_free["feature_set"]).tolist()
mae = measurement_free["mae"].to_numpy(float)
errors = np.vstack(
    [
        mae - measurement_free["mae_ci95_low"].to_numpy(float),
        measurement_free["mae_ci95_high"].to_numpy(float) - mae,
    ]
)
positions = np.arange(len(measurement_free))
figure, axis = plt.subplots(figsize=(11, 5.5))
axis.bar(positions, mae, color="#2b6f8a")
axis.errorbar(positions, mae, yerr=errors, fmt="none", color="black")
axis.set_xticks(positions, labels, rotation=30, ha="right")
axis.set(
    title="Joint future-date/unseen-site holdout — measurement-free task",
    ylabel="MAE (vehicles per 20-minute interval)",
)
axis.grid(axis="y", alpha=0.2)
figure.tight_layout()
plt.show()


A lower point estimate does not establish statistical significance: site-bootstrap intervals are wide and the holdout contains only ten inferred sites. EO features are not part of this comparison because the existing imagery covers only the selected Tahrir-area link pair rather than all analytical sites.


## Online one-step persistence

Persistence uses the immediately previous measured count only when it is exactly 20 minutes earlier on the same directional segment-day. It has a stronger information budget and fewer eligible holdout rows, so it is shown separately and must not be used to claim that the measurement-free models underperform on the same task. Gaps are never filled.


In [ ]:
online = metrics.loc[
    metrics["information_budget"] == "online_forecast_with_previous_measurement"
]
online[
    [
        "model",
        "feature_set",
        "rows",
        "inferred_sites",
        "mae",
        "mae_ci95_low",
        "mae_ci95_high",
        "rmse",
        "rmse_ci95_low",
        "rmse_ci95_high",
    ]
]


## Grouped development folds and site error heterogeneity


In [ ]:
cv_summary = (
    cv_metrics.groupby(["model", "feature_set"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
    )
    .sort_values("mean_mae")
)
cv_summary


In [ ]:
best = measurement_free.iloc[0]
best_site_errors = (
    site_errors.loc[
        (site_errors["model"] == best["model"])
        & (site_errors["feature_set"] == best["feature_set"])
    ]
    .sort_values("mae", ascending=False)
    .reset_index(drop=True)
)
best_site_errors[["inferred_site_id", "rows", "mae", "rmse", "mean_bias"]]


## Artifact and scientific-integrity checks

The validated bundle binds the official source hashes, normalized source manifest, configuration, frozen split, predictions, metrics, site/class summaries, EO context, and figures. The source labels remain separate:

- traffic: measured TfC directional link counts (2024);
- geometry: TfC-released OSM-derived survey lines (2026 OSM extract);
- EO: real Sentinel-2 L2A and WorldCover 2021 historic context;
- measured-demand signal simulation: not run because the evidence gate failed.

No complete turning matrix, signal phases, timings, pedestrian behavior, or route demand is observed. Consequently, this workflow does not relabel the existing abstract SUMO benchmark or claim counterfactual Cairo signal benefits.


In [ ]:
integrity = pd.Series(
    {
        "source counts SHA-256": provenance["source_counts_sha256"],
        "source locations SHA-256": provenance["source_locations_sha256"],
        "config SHA-256": provenance["config_sha256"],
        "split SHA-256": provenance["split_manifest_sha256"],
        "Git commit": provenance["git"]["commit"],
        "Git working tree dirty": provenance["git"]["dirty"],
        "prediction rows validated": len(predictions),
        "TfC redistribution permitted": provenance["license"][
            "redistribution_permitted"
        ],
        "measured-demand SUMO permitted": provenance["study_gate"][
            "allow_measured_demand_sumo"
        ],
    },
    name="value",
)
integrity.to_frame()
